# Free Model Gateway — Phase 1 (Colab T4)

Vertical slice: **Ollama → qwen3-8b → LiteLLM → Cloudflare Quick Tunnel**.

**Setup:** clone or upload this repo so `config/models.yaml` is visible, then **Runtime → Run all**.

Phase 1 locks the model to `qwen3-8b` (change only after Sprint 1 validation).

In [ ]:
# @title Configuration
MODEL = "qwen3-8b"  # @param ["qwen3-8b"]
CONTEXT = 32768  # @param [4096, 8192, 16384, 32768]
HOST_ID = "colab-t4-01"  # @param ["colab-t4-01", "colab-t4-02"]

# Optional: set a stable key; otherwise one is generated for this session.
# import os; os.environ["GATEWAY_API_KEY"] = "your-long-secret"

print(f"MODEL={MODEL} CONTEXT={CONTEXT} HOST_ID={HOST_ID}")

In [ ]:
# @title Locate repo + Python path
from pathlib import Path
import sys

# If you uploaded a zip, extract first or set REPO_ROOT manually:
# REPO_ROOT = Path("/content/ai_free_providers")
def _find_repo() -> Path:
    search = [Path.cwd(), *Path.cwd().parents, Path("/content")]
    for candidate in search:
        if (candidate / "config" / "models.yaml").is_file():
            return candidate
        matches = list(candidate.glob("*/config/models.yaml"))
        if matches:
            return matches[0].parent.parent
    raise FileNotFoundError(
        "Upload/clone the repo into Colab so config/models.yaml exists "
        "(e.g. /content/ai_free_providers)."
    )

REPO_ROOT = _find_repo()
sys.path.insert(0, str(REPO_ROOT / "cli"))
sys.path.insert(0, str(REPO_ROOT / "host" / "colab"))
print("REPO_ROOT=", REPO_ROOT)

In [ ]:
# @title Detect GPU
import subprocess

result = subprocess.check_output(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total",
        "--format=csv,noheader",
    ],
    text=True,
)
print(result)

In [ ]:
# @title Validate selection (registry)
from modelctl.services.registry import load_registry, validate_selection
from modelctl.services.vram import detect_gpus

registry = load_registry(REPO_ROOT / "config")
gpus = detect_gpus()
print(gpus[0])
host, model = validate_selection(
    registry,
    HOST_ID,
    MODEL,
    CONTEXT,
    available_vram_gb=gpus[0].memory_total_gb,
)
print("OK:", host.id, model.id, model.model)

In [ ]:
# @title Install → start → tunnel → READY
# Force-reload so Colab does not reuse a stale bootstrap/runtime from an older upload.
import importlib
import bootstrap
import runtime

importlib.reload(bootstrap)
importlib.reload(runtime)

print("bootstrap file:", bootstrap.__file__)
print("has install_all:", hasattr(bootstrap, "install_all"))

result = runtime.run(
    model_id=MODEL,
    context=CONTEXT,
    host_id=HOST_ID,
    repo_root=REPO_ROOT,
    skip_install=False,
    run_chat_probe=True,
)
print("endpoint:", result.endpoint)

## Verify from your PC

```bash
export URL="https://xxxxx.trycloudflare.com"   # from READY banner
export GATEWAY_API_KEY="..."                   # printed below banner

curl -sS -H "Authorization: Bearer $GATEWAY_API_KEY" "$URL/v1/models"
```